# Minimal working example using BERT (`distBERT` model)

In [1]:
# !pip install pandas torch transformers faiss-cpu
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 27.0 MB/s eta 0:00:00


## Setup

In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import faiss

## Pseudo data creation

In [39]:
# ----------------------------
# 1) Sample pseudo "data"
# ----------------------------
items = pd.DataFrame([
    {"item_id":"i1","title":"Noise Cancelling Headphones","description":"Wireless noise-cancelling headphones with 30-hour battery life","category":"electronics"},
    {"item_id":"i2","title":"Mechanical Keyboard","description":"Mechanical keyboard with RGB and hot-swappable switches","category":"electronics"},
    {"item_id":"i3","title":"Running Shoes","description":"Running shoes designed for long distance comfort and stability","category":"sports"},
    {"item_id":"i4","title":"Vegetarian Cookbook","description":"Cookbook featuring quick vegetarian recipes for busy weeknights","category":"books"},
    {"item_id":"i5","title":"Fitness Smartwatch","description":"Smartwatch with heart rate monitoring, GPS, and sleep tracking","category":"electronics"},
])

events = pd.DataFrame([
    {"user_id":"u1","item_id":"i1","ts":"2026-01-10"},
    {"user_id":"u1","item_id":"i5","ts":"2026-01-11"},
    {"user_id":"u2","item_id":"i2","ts":"2026-01-12"},
    {"user_id":"u3","item_id":"i3","ts":"2026-01-13"},
    {"user_id":"u3","item_id":"i4","ts":"2026-01-14"},
])
events["ts"] = pd.to_datetime(events["ts"])


## BERT encoder (What makes BERT work?)

In [4]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "distilbert-base-uncased"  # for illustration, 66M model

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


**Evaluate the BERT encoder**

In [30]:
# ----------------------------
# 2) BERT encode helper
# ----------------------------

# turn off gradient calculation (no back propagation in using BERT)
@torch.no_grad()
def bert_embed(texts, max_len=1024):
    batch = tokenizer(
        texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt"
    )
    batch = {k: v.to(DEVICE) for k, v in batch.items()}
    out = encoder(**batch)
    cls = out.last_hidden_state[:, 0]          # first column [CLS]-like token for classification
    emb = F.normalize(cls, dim=-1)             # normalization
    return emb.cpu().numpy().astype("float32") # size (B, 768)


## Embedding items into vectors (for comparisons)

In [6]:
# ----------------------------
# 3) Offline job: item embeddings
# ----------------------------

items["text"] = items["title"] + ". " + items["description"]
item_vecs = bert_embed(items["text"].tolist())
item_id_list = items["item_id"].tolist()

# Build ANN index (inner product works with normalized vectors)
index = faiss.IndexFlatIP(item_vecs.shape[1])
index.add(item_vecs)


## What user-specific data are there?

In [7]:
# ----------------------------
# 4) Feature builder: user text from last N clicks
# ----------------------------
def build_user_text(user_id, events, items, N=3):
    hist = (events[events["user_id"] == user_id]
            .sort_values("ts")
            .tail(N)["item_id"]
            .tolist())
    if not hist:
        return "no history"
    text = items.set_index("item_id").loc[hist, "text"].tolist()
    return " ".join(text), set(hist)


## How to recommend "similar" item with BERT?

In [31]:
# ----------------------------
# 5) "What to recommend" function
# ----------------------------
def recommend(user_id, k=3):
    user_text, seen = build_user_text(user_id, events, items, N=k)
    u = bert_embed([user_text])  # (1, 768)
    scores, idx = index.search(u, k + len(seen))  # keep track of what was seen by the user
    recs = []
    for j in idx[0]:
        iid = item_id_list[j]
        if iid not in seen:
            recs.append(iid)
        if len(recs) == k:
            break
    return recs


In [13]:

for u in ["u1","u2","u3"]:
    print(u, "->", recommend(u, k=3))


u1 -> ['i2', 'i3', 'i4']
u2 -> ['i1', 'i3', 'i5']
u3 -> ['i5', 'i1', 'i2']


In [42]:
import pandas
events = pandas.read_csv('events.csv')
items = pandas.read_csv('items.csv')

In [44]:
print(events.head())
print(items.head())

  user_id item_id                   ts   event_type       session_id   device  \
0      u1      i2  2026-01-01 07:47:00        click  s_u1_20260101_1   tablet   
1      u1     i13  2026-01-04 08:01:00         view  s_u1_20260104_2   tablet   
2      u1      i2  2026-01-07 14:28:00        click  s_u1_20260107_3   mobile   
3      u1     i13  2026-01-07 19:06:00         view  s_u1_20260107_1   tablet   
4      u1      i1  2026-01-10 08:42:00  add_to_cart  s_u1_20260110_1  desktop   

   dwell_seconds  price_at_event category_at_event  
0             56          129.00       electronics  
1              7          149.99       electronics  
2             72          129.00       electronics  
3             67          149.99       electronics  
4             46          199.99       electronics  
  item_id                        title  \
0      i1  Noise Cancelling Headphones   
1      i2          Mechanical Keyboard   
2      i3                Running Shoes   
3      i4          Vegetari

In [45]:
# itemscsv["text"] = itemscsv["title"] + ". " + itemscsv["description"]
# itemcsv_vecs = bert_embed(itemscsv["text"].tolist())
# itemcsv_id_list = itemscsv["item_id"].tolist()

# # Build ANN index (inner product works with normalized vectors)
# index = faiss.IndexFlatIP(itemcsv_vecs.shape[1])
# index.add(itemcsv_vecs)
items["text"] = items["title"] + ". " + items["description"]
item_vecs = bert_embed(items["text"].tolist())
item_id_list = items["item_id"].tolist()

# Build ANN index (inner product works with normalized vectors)
index = faiss.IndexFlatIP(item_vecs.shape[1])
index.add(item_vecs)


In [46]:

for u in ["u1","u2","u3"]:
    print(u, "->", recommend(u, k=3))

u1 -> ['i6', 'i13', 'i18']
u2 -> ['i18', 'i11', 'i6']
u3 -> ['i8', 'i4', 'i18']
